# Bronze Layer: Reverse Historical Ingestion
This notebook fetches seismic data from the USGS API year by year, running in **reverse chronological order** (from 2026 back to 2000). 
It writes raw JSONs into DBFS storage and logs the metadata into our `bronze.seismic_raw` table.

# 🏗️ Bronze Layer Architectural Design Decisions

---

### 1. Decoupled Dual-Storage Architecture (Landing vs. Bronze)
> **The Decision:** 
> We implement a clear distinction between our **Landing Zone** and our **Bronze Table**. The raw, unaltered GeoJSON responses are saved as physical files (`.json`) inside a Unity Catalog Managed Volume (`/Volumes/cr_seismic_analysis/bronze/seismic_data_raw/`). Concurrently, we use Spark to flatten this raw nested structure and load it into a structured Delta table (`bronze.seismic_raw`) using resilient `STRING` types.
> 
> **The "Why":** 
> This approach gives us the best of both worlds. The Managed Volume acts as our immutable cold storage—protecting us against upstream API schema drift and allowing us to rebuild the entire pipeline without hitting external servers. Meanwhile, the structured Bronze Delta table acts as our queryable, high-performance entry point. Storing the data as flattened rows instead of a single massive JSON string drastically reduces serialization overhead during downstream processing.

---

### 2. Natural Key Idempotency & Unified Delta MERGE
> **The Decision:** 
> Instead of using legacy transactional deletions matching file names, we leverage Delta Lake's native, highly performant `MERGE` operation executed over the unique seismic event identifier (`event_id`).
> 
> **The "Why":** 
> A sismo (earthquake) is a distinct physical event with a persistent natural key assigned by the USGS (`id`). By performing a `MERGE` on `target.event_id = source.event_id`, we guarantee absolute idempotency. If an execution processes the same year file twice, or if a seismic event overlaps between two distinct raw files, Delta Lake gracefully handles updates and avoids any duplicate records. This ensures a clean, reliable, and duplicate-free baseline before the data ever reaches the Silver layer.

---

### 3. Structural Scope Filtering with Resilient Casting
> **The Decision:** 
> During the flattening process in the Bronze stage, we project only the essential analytical columns (magnitude, coordinates, depth, place, time, and event type) and cast every single field to a lenient `STRING` data type. 
> 
> **The "Why":** 
> By selecting only the core fields, we prevent our Lakehouse from turning into a "data swamp" filled with system-specific API telemetries (e.g., station gap counts, seismic network codes, and technical URLs) that carry zero business value. However, we strictly defer all data cleaning, coordinate casting, and timezone transformations to the **Silver Layer**. Keeping everything as a resilient `STRING` in Bronze prevents the pipeline from failing if the API suddenly shifts its numeric precision or data formats.

---

### 4. Enterprise Auditability & Unity Catalog Compliance
> **The Decision:** 
> We capture metadata for operational auditability by recording the exact system ingestion timestamp and extracting the origin file path using Unity Catalog's secure `_metadata.file_path` context.
> 
> **The "Why":** 
> Standard PySpark functions like `input_file_name()` are blocked in Unity Catalog's shared governance environments due to security restrictions. Using the native `_metadata` struct ensures full compliance with Databricks security policies while giving us full lineage tracking—allowing us to trace any single seismic row back to its exact raw source JSON file.

In [0]:
import requests
import json
import time 
import os
from datetime import datetime

# 1. Parameter configuration
CURRENT_YEAR = datetime.now().year
START_YEAR = 2000
END_YEAR = CURRENT_YEAR
BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"
BRONZE_MOUNT_PATH = "/Volumes/cr_seismic_analysis/bronze/seismic_data_raw/"

# Ensure the Managed Volume directory exists
os.makedirs(BRONZE_MOUNT_PATH, exist_ok=True)

# 2. Reverse chronological ingestion loop to Managed Volume
for year in range(END_YEAR, START_YEAR - 1, -1):
    file_name = f"seismic_raw_{year}.json"
    dest_path = os.path.join(BRONZE_MOUNT_PATH, file_name)
    
    # Skip API requests for static historical years if the file already exists in the Volume
    if year != CURRENT_YEAR and os.path.exists(dest_path):
        print(f"Year {year} already exists in Volume. Skipping API request.")
        continue
        
    print(f"Downloading GeoJSON from USGS API for year {year}...")
    
    params = {
        "format": "geojson",
        "starttime": f"{year}-01-01",
        "endtime": f"{year}-12-31",
        "minlatitude": 8.0,
        "maxlatitude": 11.5,
        "minlongitude": -86.0,
        "maxlongitude": -82.5
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        
        if response.status_code == 200:
            json_data = response.json()
            
            # Persist raw JSON payload to the Managed Volume (Landing)
            with open(dest_path, "w") as f:
                json.dump(json_data, f)
            print(f"-> Successfully saved: {file_name}")
            
        else:
            print(f"⚠️ HTTP error {response.status_code} encountered for year {year}.")
            
    except Exception as e:
        print(f"❌ Connection error occurred for year {year}: {str(e)}")
        
    print("Waiting 1.5 seconds to comply with API rate-limiting...")
    time.sleep(1.5)

In [0]:
%sql
-- Verification of the files in landing
USE CATALOG cr_seismic_analysis;

-- List files using the explicit string path
LIST '/Volumes/cr_seismic_analysis/bronze/seismic_data_raw/';

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# Establish Unity Catalog database context
spark.sql("USE CATALOG cr_seismic_analysis")
spark.sql("USE SCHEMA bronze")

# 1. Read raw JSON files directly from the Managed Volume using Spark
# Spark natively infers and parallelizes JSON folder paths
df_raw_files = spark.read.option("multiline", "true").json(f"{BRONZE_MOUNT_PATH}/*.json")

# 2. Flatten the GeoJSON nested structure using PySpark
# Explode 'features' array and extract key properties as resilient string types
df_flattened = df_raw_files \
    .select(
        F.explode("features").alias("event"), 
        F.col("_metadata.file_path").alias("source_file")  # Correct Unity Catalog syntax
    ) \
    .select(
        F.col("event.id").cast("string").alias("event_id"),
        F.col("event.properties.mag").cast("string").alias("magnitude"),
        F.col("event.properties.place").cast("string").alias("place"),
        F.col("event.properties.time").cast("string").alias("event_timestamp_raw"), # Epoch ms from API
        F.col("event.properties.type").cast("string").alias("event_type"),
        F.col("event.geometry.coordinates").getItem(0).cast("string").alias("longitude"),
        F.col("event.geometry.coordinates").getItem(1).cast("string").alias("latitude"),
        F.col("event.geometry.coordinates").getItem(2).cast("string").alias("depth"),
        F.col("source_file") # Traceability metadata
    ) \
    .withColumn("ingestion_timestamp", F.current_timestamp())

# 3. Idempotent write using Delta MERGE on unique natural key 'event_id'
target_table = "seismic_raw"

# Check if target table already exists to perform Delta MERGE
if spark.catalog.tableExists(f"cr_seismic_analysis.bronze.{target_table}"):
    delta_target = DeltaTable.forName(spark, f"cr_seismic_analysis.bronze.{target_table}")
    
    delta_target.alias("target") \
        .merge(
            df_flattened.alias("source"),
            "target.event_id = source.event_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    print("Delta MERGE executed successfully. Data synchronized in Bronze table.")
else:
    # Perform initial table creation if it does not exist
    df_flattened.write.format("delta").mode("overwrite").saveAsTable(target_table)
    print(f"Delta table '{target_table}' initialized successfully with first payload load.")